In [4]:
# 1. Install dependencies
!pip -q install -U langgraph langchain langchain-openai beautifulsoup4 requests pydantic

# Optional: restart the Colab runtime only if Colab asks for it after installation.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
# 2. Imports and OpenAI configuration

from google.colab import userdata
import os, re, json, math, hashlib, time
from typing import TypedDict, List, Dict, Any
from urllib.parse import quote_plus, urljoin

import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY. Add it in Colab: Secrets -> OPENAI_API_KEY.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Requested model
MODEL_NAME = "gpt-5.4-nano"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.1,
    api_key=OPENAI_API_KEY,
)

print("LLM configured:", MODEL_NAME)


LLM configured: gpt-5.4-nano


In [7]:
# 3. Data models

class CandidateProfile(BaseModel):
    name: str = "Candidate"
    target_roles: List[str] = Field(default_factory=list)
    skills: List[str] = Field(default_factory=list)
    years_experience: float = 0.0
    education: List[str] = Field(default_factory=list)
    languages: List[str] = Field(default_factory=list)
    location: str = ""
    work_preference: str = ""
    salary_expectation: str = ""
    must_have: List[str] = Field(default_factory=list)
    nice_to_have: List[str] = Field(default_factory=list)

class Job(BaseModel):
    title: str
    company: str = "Unknown"
    location: str = ""
    work_mode: str = ""
    skills: List[str] = Field(default_factory=list)
    description: str = ""
    url: str = ""
    source: str = ""

class Match(BaseModel):
    job: Job
    score: float
    matched_skills: List[str] = Field(default_factory=list)
    missing_skills: List[str] = Field(default_factory=list)
    reasons: List[str] = Field(default_factory=list)

class WorkflowState(TypedDict, total=False):
    user_request: str
    candidate: Dict[str, Any]
    jobs: List[Dict[str, Any]]
    matches: List[Dict[str, Any]]
    human_feedback: str
    final_answer: str
    trace: List[str]


In [8]:
# 4. Utility functions and deterministic fallback data

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0 Safari/537.36"
}

DEMO_JOBS = [
    Job(title="Junior Python Developer", company="Demo Software", location="Sofia",
        work_mode="Hybrid", skills=["Python", "SQL", "Git", "REST"], source="demo",
        url="https://dev.bg/job-listings/"),
    Job(title="Junior Data Analyst", company="Demo Analytics", location="Sofia",
        work_mode="Hybrid", skills=["Python", "SQL", "Excel", "Power BI"], source="demo",
        url="https://www.jobs.bg/"),
    Job(title="QA Automation Engineer", company="Demo Tech", location="Sofia",
        work_mode="Remote", skills=["Python", "Selenium", "API", "Git"], source="demo",
        url="https://dev.bg/job-listings/"),
    Job(title="Backend Developer", company="Demo Cloud", location="Sofia",
        work_mode="Remote", skills=["Python", "FastAPI", "PostgreSQL", "Docker"], source="dev.bg",
        url="https://dev.bg/job-listings/"),
    Job(title="Software Engineer", company="Demo Systems", location="Plovdiv",
        work_mode="Hybrid", skills=["Java", "SQL", "Git", "Docker"], source="jobs.bg",
        url="https://www.jobs.bg/"),
]

def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", x or "").strip()

def unique_keep_order(items):
    seen, out = set(), []
    for x in items:
        k = x.lower().strip()
        if k and k not in seen:
            seen.add(k); out.append(x.strip())
    return out

def extract_skills(text: str) -> List[str]:
    catalog = [
        "Python","Java","C#",".NET","JavaScript","TypeScript","React","Angular","Vue",
        "SQL","PostgreSQL","MySQL","MongoDB","Docker","Kubernetes","AWS","Azure","GCP",
        "Git","Linux","FastAPI","Django","Flask","Selenium","Cypress","REST","GraphQL",
        "Power BI","Excel","Pandas","NumPy","TensorFlow","PyTorch","LangChain","LangGraph",
        "OpenAI","Machine Learning","AI","Data Science","HTML","CSS"
    ]
    low = text.lower()
    return [s for s in catalog if s.lower() in low]

def demo_or_live(url, source, limit=12):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        text = clean_text(soup.get_text(" ", strip=True))
        if len(text) < 300:
            raise ValueError("Page content too small")
        return r.text, soup
    except Exception as e:
        return None, str(e)


In [9]:
# 5. TOOL 1 — DEV.BG public listing search

@tool
def search_devbg_jobs(query: str, location: str = "Sofia", limit: int = 12) -> List[dict]:
    """Search public DEV.BG listings and return normalized job records."""
    url = "https://dev.bg/job-listings/"
    html, result = demo_or_live(url, "dev.bg", limit)
    jobs = []

    if html:
        soup = result
        # DEV.BG pages contain listing cards/headings. We use broad extraction so
        # small site markup changes do not break the notebook.
        for card in soup.select("article, .job-listing, .job-listing-item, .card"):
            txt = clean_text(card.get_text(" ", strip=True))
            if len(txt) < 25:
                continue
            skills = extract_skills(txt)
            if query and query.lower() not in txt.lower() and not any(
                q.lower() in txt.lower() for q in query.split()
            ):
                continue
            title_el = card.find(["h2","h3","h4","a"])
            title = clean_text(title_el.get_text(" ", strip=True)) if title_el else txt[:90]
            link = title_el.get("href") if title_el and title_el.name == "a" else ""
            jobs.append(Job(
                title=title[:160],
                company="DEV.BG listing",
                location=location,
                skills=skills,
                description=txt[:700],
                url=urljoin(url, link) if link else url,
                source="DEV.BG"
            ).model_dump())
            if len(jobs) >= limit:
                break

    if not jobs:
        # Safe fallback: still a real tool execution, but clearly labeled as demo data.
        q = query.lower()
        for j in DEMO_JOBS:
            if not q or q in j.title.lower() or any(q in s.lower() for s in j.skills):
                jobs.append(j.model_copy(update={"source": "DEV.BG (demo fallback)"}).model_dump())
        if not jobs:
            jobs = [j.model_dump() for j in DEMO_JOBS[:limit]]

    return jobs[:limit]


In [10]:
# 6. TOOL 2 — JOBS.BG public listing search

@tool
def search_jobsbg_jobs(query: str, location: str = "Sofia", limit: int = 12) -> List[dict]:
    """Search the public JOBS.BG search page and return normalized job records."""
    search_url = (
        "https://www.jobs.bg/front_job_search.php"
        f"?frompage=0&term={quote_plus(query)}"
    )
    html, result = demo_or_live(search_url, "jobs.bg", limit)
    jobs = []

    if html:
        soup = result
        # Extract candidate links and nearby text from the search result page.
        for a in soup.find_all("a", href=True):
            title = clean_text(a.get_text(" ", strip=True))
            if len(title) < 8 or len(title) > 180:
                continue
            parent = a.parent
            context = clean_text(parent.get_text(" ", strip=True)) if parent else title
            if len(context) < 20:
                continue
            skills = extract_skills(context)
            if query and query.lower() not in (title + " " + context).lower() and not skills:
                continue
            jobs.append(Job(
                title=title,
                company="JOBS.BG listing",
                location=location,
                skills=skills,
                description=context[:700],
                url=urljoin(search_url, a["href"]),
                source="JOBS.BG"
            ).model_dump())
            if len(jobs) >= limit:
                break

    if not jobs:
        q = query.lower()
        for j in DEMO_JOBS:
            if not q or q in j.title.lower() or any(q in s.lower() for s in j.skills):
                jobs.append(j.model_copy(update={"source": "JOBS.BG (demo fallback)"}).model_dump())
        if not jobs:
            jobs = [j.model_dump() for j in DEMO_JOBS[:limit]]

    return jobs[:limit]


In [11]:
# 7. TOOL 3 — deterministic match scorer

@tool
def score_job_matches(candidate_json: str, jobs_json: str, top_k: int = 8) -> List[dict]:
    """Score jobs against a candidate profile using explainable deterministic rules."""
    candidate = CandidateProfile.model_validate_json(candidate_json)
    jobs = [Job.model_validate(x) for x in json.loads(jobs_json)]

    candidate_skills = {s.lower() for s in candidate.skills + candidate.must_have + candidate.nice_to_have}
    target_roles = [x.lower() for x in candidate.target_roles]
    results = []

    for job in jobs:
        job_skills = {s.lower() for s in job.skills}
        matched = sorted(candidate_skills & job_skills)
        missing = sorted(job_skills - candidate_skills)

        title_bonus = 0
        title_low = job.title.lower()
        for role in target_roles:
            if any(tok in title_low for tok in role.split() if len(tok) > 2):
                title_bonus = 15
                break

        skill_score = 55 * (len(matched) / max(1, len(job_skills)))
        location_bonus = 10 if (not candidate.location or candidate.location.lower() in job.location.lower()) else 0
        score = min(100, round(skill_score + title_bonus + location_bonus, 1))

        reasons = []
        if matched:
            reasons.append("Matched skills: " + ", ".join(matched))
        if title_bonus:
            reasons.append("Job title aligns with a target role")
        if location_bonus:
            reasons.append("Location matches the candidate preference")
        if missing:
            reasons.append("Potential gaps: " + ", ".join(missing[:5]))

        results.append(Match(
            job=job,
            score=score,
            matched_skills=matched,
            missing_skills=missing[:8],
            reasons=reasons
        ).model_dump())

    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


In [12]:
# 8. Agent prompts

PROFILE_SYSTEM = """
You are Candidate Profiler, the first agent in a job-application workflow.
Turn the user's CV/request into a structured candidate profile.
Never invent experience, education, skills, salary, location, or language level.
If information is missing, leave it empty or use a conservative default.
Return ONLY valid JSON matching the CandidateProfile schema.
"""

RESEARCH_SYSTEM = """
You are Job Researcher. Find relevant jobs for the candidate using the supplied
job-board tools. Prefer real listing information. Do not invent companies,
requirements, salaries, locations, or URLs. Return a compact JSON list of jobs.
"""

MATCH_SYSTEM = """
You are Match Analyst. Explain which jobs fit the candidate and why.
Use the deterministic scoring tool as the numerical baseline, then use your
language reasoning to explain strengths, gaps, and practical next steps.
Never claim a candidate has a skill that is not present in the profile.
"""

def llm_json(prompt: str) -> dict:
    response = llm.invoke(prompt)
    raw = response.content if isinstance(response.content, str) else str(response.content)
    raw = re.sub(r"^```json\s*|\s*```$", "", raw.strip(), flags=re.I)
    return json.loads(raw)

def profile_agent(state: WorkflowState):
    prompt = PROFILE_SYSTEM + "\n\nUSER REQUEST:\n" + state["user_request"]
    schema_hint = CandidateProfile.model_json_schema()
    prompt += "\n\nJSON SCHEMA:\n" + json.dumps(schema_hint, ensure_ascii=False)
    data = llm_json(prompt)
    profile = CandidateProfile.model_validate(data)
    trace = state.get("trace", []) + ["Agent 1: Candidate Profiler completed."]
    return {"candidate": profile.model_dump(), "trace": trace}

def researcher_agent(state: WorkflowState):
    c = CandidateProfile.model_validate(state["candidate"])
    query = c.target_roles[0] if c.target_roles else (c.skills[0] if c.skills else "software developer")
    dev = search_devbg_jobs.invoke({"query": query, "location": c.location or "Sofia", "limit": 10})
    jobs = search_jobsbg_jobs.invoke({"query": query, "location": c.location or "Sofia", "limit": 10})

    # De-duplicate by normalized title/company/url
    combined = dev + jobs
    seen, unique = set(), []
    for j in combined:
        key = (j.get("title","").lower(), j.get("company","").lower(), j.get("url",""))
        if key not in seen:
            seen.add(key); unique.append(j)

    trace = state.get("trace", []) + [
        f"Agent 2: Job Researcher used DEV.BG ({len(dev)} results) and JOBS.BG ({len(jobs)} results)."
    ]
    return {"jobs": unique[:20], "trace": trace}

def matcher_agent(state: WorkflowState):
    c = CandidateProfile.model_validate(state["candidate"])
    matches = score_job_matches.invoke({
        "candidate_json": c.model_dump_json(),
        "jobs_json": json.dumps(state["jobs"], ensure_ascii=False),
        "top_k": 8
    })
    trace = state.get("trace", []) + ["Agent 3: Match Analyst scored and ranked the jobs."]
    return {"matches": matches, "trace": trace}


In [22]:
# 9. HITL and final answer nodes

def human_review_node(state: WorkflowState):
    top = state.get("matches", [])[:5]

    review_text = "I've prepared a shortlist of the most relevant jobs for you.\n\n"

    for i, m in enumerate(top, 1):
        job = m["job"]

        review_text += (
            f"{i}. {job['title']} — {job['company']}\n"
            f"   Match: {m['score']}%\n"
            f"   Location: {job.get('location') or 'Not specified'}\n"
            f"   Work mode: {job.get('work_mode') or 'Not specified'}\n"
            f"   Matched skills: "
            f"{', '.join(m.get('matched_skills', [])) or 'None identified'}\n\n"
        )

    review_text += (
        "Please review the shortlist before I prepare the final recommendations.\n\n"
        "You can reply with:\n"
        "• approve — to continue with this shortlist\n"
        "• or tell me what you would like to change, for example "
        "\"prioritize remote jobs\" or \"keep only junior positions\"."
    )

    feedback = interrupt(review_text)

    return {
        "human_feedback": str(feedback),
        "trace": state.get("trace", []) + ["HITL: human feedback received."]
    }

def final_agent(state: WorkflowState):
    c = CandidateProfile.model_validate(state["candidate"])
    matches = state.get("matches", [])
    feedback = state.get("human_feedback", "")

    prompt = MATCH_SYSTEM + f"""

CANDIDATE:
{c.model_dump_json(indent=2)}

RANKED MATCHES:
{json.dumps(matches, ensure_ascii=False, indent=2)}

HUMAN FEEDBACK:
{feedback}

Create the final response in Bulgarian.
For each recommended job include: rank, title, company, fit score,
matched skills, important gaps, source and URL.
Then add:
- Why these jobs are a fit
- What to improve in the CV
- 3 concrete application tips
Do not invent missing job facts. If a field is unavailable, say "не е посочено".
"""

    response = llm.invoke(prompt)
    answer = response.content if isinstance(response.content, str) else str(response.content)
    trace = state.get("trace", []) + ["Final Agent: final recommendations generated."]
    return {"final_answer": answer, "trace": trace}

def build_graph():
    builder = StateGraph(WorkflowState)
    builder.add_node("candidate_profiler", profile_agent)
    builder.add_node("job_researcher", researcher_agent)
    builder.add_node("match_analyst", matcher_agent)
    builder.add_node("human_review", human_review_node)
    builder.add_node("final_agent", final_agent)

    builder.add_edge(START, "candidate_profiler")
    builder.add_edge("candidate_profiler", "job_researcher")
    builder.add_edge("job_researcher", "match_analyst")
    builder.add_edge("match_analyst", "human_review")
    builder.add_edge("human_review", "final_agent")
    builder.add_edge("final_agent", END)

    return builder.compile(checkpointer=MemorySaver())

def execute_workflow(user_request: str):
    """Required core function: accepts exactly one user_request string."""
    graph = build_graph()
    thread_id = "cv-job-" + hashlib.sha256(user_request.encode()).hexdigest()[:12]
    config = {"configurable": {"thread_id": thread_id}}

    first = graph.invoke(
        {"user_request": user_request, "trace": []},
        config=config
    )

    if "__interrupt__" not in first:
        return first

    interrupt_message = first["__interrupt__"][0].value

    print("\n" + "=" * 70)
    print("REVIEW REQUIRED")
    print("=" * 70)
    print(interrupt_message)

    feedback = input("\nYour response: ").strip()

    resumed = graph.invoke(Command(resume=feedback), config=config)
    print("\n=== WORKFLOW TRACE ===")
    for item in resumed.get("trace", []):
        print("-", item)

    print("\n=== FINAL RESULT ===\n")
    print(resumed.get("final_answer", "No final answer produced."))
    return resumed


In [18]:
# ============================================================
# TEST CASE 1 — Python Developer
# ============================================================

test_case_1 = """
I am looking for a Python Developer position in Sofia.
I have 2 years of experience with Python, FastAPI, SQL and Git.
I prefer hybrid or remote work.
Find suitable job opportunities and rank them based on my profile.
"""

result_1 = execute_workflow(test_case_1)

print(result_1)


=== HUMAN-IN-THE-LOOP REVIEW ===
{
  "message": "Human review required before final recommendations. Approve the ranking or provide feedback (e.g. 'prefer remote', 'only junior roles', 'ignore jobs below 70').",
  "top_matches": [
    {
      "title": "Junior Python Developer",
      "company": "Demo Software",
      "score": 66.2,
      "source": "DEV.BG (demo fallback)"
    }
  ]
}

Your decision (e.g. 'approve' or detailed revision feedback): approve

=== WORKFLOW TRACE ===
- Agent 1: Candidate Profiler completed.
- Agent 2: Job Researcher used DEV.BG (1 results) and JOBS.BG (1 results).
- Agent 3: Match Analyst scored and ranked the jobs.
- HITL: human feedback received.
- Final Agent: final recommendations generated.

=== FINAL RESULT ===

## Препоръчани обяви (най-добър мач)

### 1) Rank #1 — Junior Python Developer — Demo Software
- **Fit score (детерминистичен baseline): 66.2**
- **Локация/работа:** София, **Hybrid**
- **Matched skills (от профила):** Python, SQL, Git  
- **Im

In [23]:
# ============================================================
# TEST CASE 2 — Junior Backend Developer
# ============================================================

test_case_2 = """
I am a Junior Backend Developer looking for my first professional
opportunity in Bulgaria.

My main skills are Python, REST APIs, SQL, Git and basic Docker.
I am interested in junior backend or Python developer positions.

Search for suitable jobs and explain why each position matches my profile.
"""

result_2 = execute_workflow(test_case_2)

print(result_2)


REVIEW REQUIRED
I've prepared a shortlist of the most relevant jobs for you.

1. Junior Python Developer — Demo Software
   Match: 66.2%
   Location: Sofia
   Work mode: Hybrid
   Matched skills: git, python, sql

2. Junior Data Analyst — Demo Analytics
   Match: 52.5%
   Location: Sofia
   Work mode: Hybrid
   Matched skills: python, sql

3. Backend Developer — Demo Cloud
   Match: 38.8%
   Location: Sofia
   Work mode: Remote
   Matched skills: python

4. QA Automation Engineer — Demo Tech
   Match: 37.5%
   Location: Sofia
   Work mode: Remote
   Matched skills: git, python

5. Software Engineer — Demo Systems
   Match: 37.5%
   Location: Plovdiv
   Work mode: Hybrid
   Matched skills: git, sql

Please review the shortlist before I prepare the final recommendations.

You can reply with:
• approve — to continue with this shortlist
• or tell me what you would like to change, for example "prioritize remote jobs" or "keep only junior positions".

Your response: approve

=== WORKFLOW TR

In [ ]:
# ============================================================
# TEST CASE 3 — Java Developer
# ============================================================

test_case_3 = """
I am a Java Developer with 3 years of experience.
My main technologies are Java, Spring Boot, REST APIs, PostgreSQL,
Docker and Git.

I am looking for Backend Java Developer positions in Sofia.
Search DEV.BG and JOBS.BG and select the most relevant opportunities.
"""

result_3 = execute_workflow(test_case_3)

print(result_3)

In [ ]:
# ============================================================
# TEST CASE 4 — Data Analyst
# ============================================================

test_case_4 = """
I am looking for a Data Analyst position.

I have experience with Python, SQL, Excel, Power BI and basic statistics.
I prefer positions in Sofia, but remote opportunities are also acceptable.

Find relevant job advertisements and rank them according to my skills
and experience.
"""

result_4 = execute_workflow(test_case_4)

print(result_4)

In [ ]:
# ============================================================
# TEST CASE 5 — React Developer
# ============================================================

test_case_5 = """
I am a Frontend Developer with 2 years of experience.

My main skills are JavaScript, TypeScript, React, HTML, CSS and Git.
I am looking for React Developer or Frontend Developer positions
in Sofia or remote.

Search for suitable jobs and explain the match between my profile
and each job.
"""

result_5 = execute_workflow(test_case_5)

print(result_5)